[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LBDillon/Glycan-occupancy-analysis/blob/main/notebooks/esmc_sequence_only.ipynb)

# ESMC 300M — the sequence-only arm

Scores N-glycosylation sequons with **ESMC 300M**, a masked language model that
sees no structure at all.

## Why this is a separate notebook

`fair-esm` (ESM-IF1) and EvolutionaryScale's `esm` (ESMC, ESM3) both install a
top-level package named **`esm`**. Installing one shadows the other — not a
version conflict that can be pinned around, the same import name. So ESMC cannot
share a runtime with `esm_if_and_mpnn_gpu.ipynb`.

## What it is for

ProteinMPNN and ESM-IF both condition on a backbone. ESMC conditions only on
surrounding sequence, so it answers a question neither can:

> Does sequence context alone distinguish occupied sequons from structurally
> matched sequons carrying no glycan?

It is the control the structure-conditioned results need. If ESMC reproduces
their effect, the effect need not be structural. If it does not, they are doing
work sequence alone cannot.

Scored on the chain sequence `_parse_chains` reads — not the full UniProt
sequence — so model indices, scoreability and the matched pairs stay identical to
the other models and the comparison is like-for-like.

## Two masking schemes, two estimands

| Mode | Reads | Role |
|---|---|---|
| `single` | `P(residue at i \| every other native residue)` | primary |
| `joint` | all three sequon positions masked together | sensitivity |

`joint` exists because `single` leaves a confound: masking only the +2 residue
still shows the model a native asparagine two positions upstream, and N-X-S/T is
a heavily learned motif, so it can infer S/T from the N. On 13 dataset sites that
is worth ~0.34 log-odds of the score. It inflates both arms of a matched pair so
it largely cancels in the paired contrast, but it compresses dynamic range.

**The same confound applies to ProteinMPNN and ESM-IF.** Neither has a joint
variant yet.

## 1. Runtime

ESMC 300M is small; a GPU helps but CPU is workable (~1.6 s/site under `single`).

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU — CPU is fine here")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Install the ESMC SDK

`--no-deps` throughout is deliberate: the SDK declares `torchtext`, which is dead
against modern torch and is not imported by ESMC. The pins are the ones its
`transformers` requirement actually enforces.

In [ ]:
import subprocess, sys

def pip(*a):
    print("$ pip install", *a, flush=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps",*a], check=True)

pip("esm==3.2.2")
pip("huggingface_hub<1.0", "tokenizers>=0.21,<0.22", "transformers<4.48.2")
pip("tenacity","httpx","zstd","msgpack-numpy","cloudpathlib","brotli","attrs",
    "einops","regex","safetensors","biotite","biopython")

import importlib
for m in ("esm.models.esmc",):
    importlib.import_module(m); print(f"{m} OK")

## 3. Code and bundle

ESMC needs the structures only to read each chain's **sequence**, but it needs
the same bundle so its indices match the other models exactly.

In [ ]:
import os, subprocess, getpass
from pathlib import Path

REPO    = "github.com/LBDillon/Glycan-occupancy-analysis.git"
BRANCH  = "main"
PRIVATE = False

CHECKOUT = Path('/content/glycan-occupancy-analysis')
if not CHECKOUT.exists():
    url = f"https://{REPO}"
    if PRIVATE:
        url = f"https://{getpass.getpass('GitHub token (hidden): ').strip()}@{REPO}"
    subprocess.run(["git","clone","--depth","1","-b",BRANCH,url,str(CHECKOUT)], check=True)

NESTED = CHECKOUT/'analysis'/'experimental_glycosylation_sites'
MODULE = CHECKOUT if (CHECKOUT/'src'/'experimental_glycosylation_sites').is_dir() else NESTED
assert (MODULE/'pipeline'/'07_score.py').exists(), f"module not found under {CHECKOUT}"
os.chdir(MODULE); print("module:", MODULE)

from google.colab import drive
drive.mount('/content/drive')
RESULTS = Path('/content/drive/MyDrive/sugarfix/results'); RESULTS.mkdir(parents=True, exist_ok=True)

RELEASE_TAG = "bundle-2026-08-20"
BUNDLE_TAR  = Path('/content/bundle.tar')
url = f"https://github.com/LBDillon/Glycan-occupancy-analysis/releases/download/{RELEASE_TAG}/colab_bundle.tar"
if not (BUNDLE_TAR.exists() and BUNDLE_TAR.stat().st_size/1e9 >= 0.4):
    subprocess.run(["wget","-q","--show-progress","-O",str(BUNDLE_TAR),url])
assert BUNDLE_TAR.stat().st_size/1e9 >= 0.4, f"bundle download failed — is {url} published and the repo public?"
print(f"bundle {BUNDLE_TAR.stat().st_size/1e9:.2f} GB")

## 4. Unpack

In [ ]:
import tarfile, gzip, shutil, time

work = Path('/content/bundle'); need = work/'structures'
if not (need.is_dir() and any(need.iterdir())):
    shutil.rmtree(work, ignore_errors=True); work.mkdir(parents=True, exist_ok=True)
    with tarfile.open(BUNDLE_TAR) as tar:
        tar.extractall(work, filter='data')
assert need.is_dir(), f"no structures/ under {work}"

pdb_dir = MODULE/'data'/'cache'/'pdb'; pdb_dir.mkdir(parents=True, exist_ok=True)
gz = list(need.glob('*.gz')); t0=time.time()
for i,p in enumerate(gz,1):
    target = pdb_dir/p.stem
    if not target.exists():
        with gzip.open(p,'rb') as fh, open(target,'wb') as out: shutil.copyfileobj(fh,out)
    if i % 400 == 0: print(f"  {i}/{len(gz)} ({time.time()-t0:.0f}s)", flush=True)
for sub in ('manifests','matching'):
    dest = MODULE/'results'/sub; dest.mkdir(parents=True, exist_ok=True)
    for p in (work/sub).glob('*.csv'):
        if not (dest/p.name).exists(): shutil.copyfile(p, dest/p.name)
print(f"structures: {len(list(pdb_dir.glob('*')))} (expect 1824)")

## 5. Preflight

The token offset is **round-tripped through the tokenizer**, not assumed. That is
the check the ProteinMPNN alphabet defect went undetected for: an assumption
about how a model encodes its own input is not a fact until something reproduces
the input from it.

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore'); os.environ.setdefault('KMP_DUPLICATE_LIB_OK','TRUE')
sys.path.insert(0,'src')
import pandas as pd

from experimental_glycosylation_sites.adapters.base import SequenceDesigner, SequonScorer
from experimental_glycosylation_sites.runner_support import build_adapter

for mode in ('single','joint'):
    a = build_adapter('esmc', 'cpu', mask_mode=mode)
    print(f"  {mode:7s} SequonScorer={isinstance(a,SequonScorer)} "
          f"SequenceDesigner={isinstance(a,SequenceDesigner)}  {a.describe()}")

# offset round-trip against the real tokenizer
from experimental_glycosylation_sites.esmc_scoring import _assert_token_offset, load_model
model, tok = load_model('cuda' if torch.cuda.is_available() else 'cpu')
_assert_token_offset(tok)
print("\ntoken offset round-trips: OK")

# and the indices land on the manifest's sequons
from experimental_glycosylation_sites.esmc_scoring import chain_sequence
from experimental_glycosylation_sites.runner_support import structure_paths
paths = structure_paths()
man = pd.read_csv('results/manifests/candidate_manifest_dataset.csv', low_memory=False)
checked = 0
for r in man.itertuples(index=False):
    p = paths.get(str(r.structure_pdb_id).upper())
    if p is None: continue
    seq = chain_sequence(p, r.structure_chain_id, str(r.structure_pdb_id))
    idx = (int(r.n_model_index), int(r.plus1_model_index), int(r.plus2_model_index))
    assert "".join(seq[i] for i in idx) == r.triplet, f"index mismatch at {r.accession}:{r.position}"
    checked += 1
    if checked >= 25: break
print(f"sequon indices verified on {checked} sites")
print("\npreflight OK")

## 6. Score

Six runs: two masking schemes across the dataset and the two matched control
pools. `single` costs ~1.6 s/site on CPU (three masked variants per sequon,
batched per chain), `joint` ~0.6 s/site — cheaper than either inverse-folding
model. Every run resumes.

In [ ]:
import subprocess, sys, time

SETS = [('dataset','candidate_manifest_dataset'),
        ('controls','manifest_matched_controls'),
        ('secretory','manifest_matched_secretory')]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def run(*args):
    t0=time.time(); print("$", " ".join(map(str,args)), flush=True)
    r = subprocess.run([sys.executable, *map(str,args)], text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-2000:]); print(f"[{(time.time()-t0)/60:.1f} min]\n", flush=True)
    if r.returncode: raise RuntimeError(f"FAILED: {args}")

for mode in ('single','joint'):
    for tag, manifest in SETS:
        run('pipeline/07_score.py',
            f'results/manifests/{manifest}.csv',
            RESULTS/f'scores_{tag}_esmc_{mode}.csv',
            '--model','esmc','--mask-mode',mode,'--device',DEVICE)

## 7. What came back

In [ ]:
import pandas as pd
for f in sorted(RESULTS.glob('*esmc*.csv')):
    if f.stat().st_size < 50: continue
    d = pd.read_csv(f, low_memory=False)
    v = d.conditional_sequon_score
    print(f"{f.name:44s} {len(d):5d} rows  mean {v.mean():+.3f}  sd {v.std():.3f}")

## Back on the laptop

Copy the files down, then run the model-agnostic analysis. `09_analyse_scores.py`
hard-codes `scores_dataset.csv` / `scores_controls.csv` / `scores_secretory.csv`,
so either rename the ESMC files over those (keeping the originals) or edit the
paths — otherwise the scores are computed and then not used.

Report `single` as primary and `joint` as a sensitivity. They are **different
estimands**, not competing estimates of one, so do not average them and do not
pick the better-looking one.

Full detail: [`docs/third_model_esmc.md`](../docs/third_model_esmc.md).